# Módulo VII — *Deep Learning*

Código do livro *Ciência de Dados com Python* (Howard Roatti). Explicações no livro; repositório: https://github.com/howardroatti/roatti-ciencia-de-dados-python

In [ ]:
import sys, time, tempfile, os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, "..")   # para importar src/ a partir de codigo/notebooks/
from src.dados_sinteticos_m7 import (serie_senoide, janelas, tarefa_memoria,
                                     amostras_gaussiana)

torch.manual_seed(42)
print("PyTorch", torch.__version__)

## Do neurônio à rede

In [ ]:
def sigmoide(z): return 1 / (1 + np.exp(-z))
def relu(z): return np.maximum(0, z)
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))   # subtrair o máximo evita overflow
    return e / e.sum(axis=-1, keepdims=True)

z = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])
print("sigmoide:", sigmoide(z).round(3))
print("tanh    :", np.tanh(z).round(3))
print("ReLU    :", relu(z))
print("softmax :", softmax(np.array([2.0, 1.0, 0.1])).round(3))

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)     # XOR: não é linearmente separável

In [ ]:
def inicializar(rng, entrada=2, oculta=4):
    return {"W1": rng.normal(0, 1, (entrada, oculta)), "b1": np.zeros((1, oculta)),
            "W2": rng.normal(0, 1, (oculta, 1)),       "b2": np.zeros((1, 1))}

def frente(P, X):
    z1 = X @ P["W1"] + P["b1"];  a1 = np.tanh(z1)
    z2 = a1 @ P["W2"] + P["b2"]; p = sigmoide(z2)
    return a1, p

def perda_bce(p, y):
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def gradientes(P, X, y):
    a1, p = frente(P, X)
    dz2 = (p - y) / len(X)                  # derivada da perda em relação a z2
    dW2, db2 = a1.T @ dz2, dz2.sum(axis=0, keepdims=True)
    dz1 = (dz2 @ P["W2"].T) * (1 - a1 ** 2) # regra da cadeia; tanh'(z) = 1 - tanh(z)^2
    dW1, db1 = X.T @ dz1, dz1.sum(axis=0, keepdims=True)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

In [ ]:
P = inicializar(np.random.default_rng(42))
g = gradientes(P, X, y)

eps = 1e-6                                   # diferença finita no peso W1[0, 0]
P["W1"][0, 0] += eps;  mais = perda_bce(frente(P, X)[1], y)
P["W1"][0, 0] -= 2 * eps; menos = perda_bce(frente(P, X)[1], y)
P["W1"][0, 0] += eps
print("gradiente analítico:", g["W1"][0, 0])
print("gradiente numérico :", (mais - menos) / (2 * eps))

In [ ]:
P = inicializar(np.random.default_rng(42))
lr = 0.5
for epoca in range(3001):
    g = gradientes(P, X, y)
    for nome in P:
        P[nome] -= lr * g[nome]              # passo do gradiente descendente
    if epoca % 1000 == 0:
        print(f"época {epoca:4d}  perda = {perda_bce(frente(P, X)[1], y):.4f}")

print("probabilidades:", frente(P, X)[1].ravel().round(3))

## PyTorch na prática

In [ ]:
a = torch.arange(6.0).reshape(2, 3)
print(a, a.shape, a.dtype)
print("numpy -> tensor:", torch.from_numpy(np.array([1.0, 2.0])))
print("soma por coluna:", a.sum(dim=0), "| produto matricial:", (a @ a.T).tolist())

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
f = x ** 3 + 2 * x          # df/dx = 3x^2 + 2
f.backward()
print("f(2) =", f.item(), "| df/dx(2) =", x.grad.item())

In [ ]:
digitos = load_digits()
X_d, y_d = digitos.data, digitos.target
print("imagens:", X_d.shape, "| pixels entre", X_d.min(), "e", X_d.max())

X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.25,
                                          stratify=y_d, random_state=42)
esc = StandardScaler().fit(X_tr)              # ajusta SÓ no treino
Xtr_t = torch.tensor(esc.transform(X_tr), dtype=torch.float32)
Xte_t = torch.tensor(esc.transform(X_te), dtype=torch.float32)
ytr_t = torch.tensor(y_tr, dtype=torch.long)
yte_t = torch.tensor(y_te, dtype=torch.long)
print("treino:", tuple(Xtr_t.shape), "| teste:", tuple(Xte_t.shape))

In [ ]:
fig, eixos = plt.subplots(2, 8, figsize=(9, 2.8))
for ax, img, r in zip(eixos.ravel(), digitos.images, digitos.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(int(r)); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self, entrada=64, oculta=64, saida=10, p_dropout=0.0):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(entrada, oculta), nn.ReLU(), nn.Dropout(p_dropout),
            nn.Linear(oculta, oculta), nn.ReLU(), nn.Dropout(p_dropout),
            nn.Linear(oculta, saida))          # logits; o softmax fica dentro da perda
    def forward(self, x):
        return self.rede(x)

torch.manual_seed(42)
modelo = MLP()
print(modelo)
print("parâmetros:", sum(p.numel() for p in modelo.parameters()))

In [ ]:
torch.manual_seed(42)
modelo = MLP()
perda_fn = nn.CrossEntropyLoss()
otimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
carga = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=64, shuffle=True,
                   generator=torch.Generator().manual_seed(42))

def avaliar(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        logits = modelo(X)
        return nn.functional.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

for epoca in range(1, 21):
    modelo.train()
    for xb, yb in carga:
        otimizador.zero_grad()              # 1. zera gradientes acumulados
        perda = perda_fn(modelo(xb), yb)    # 2. frente + perda
        perda.backward()                    # 3. retropropagação
        otimizador.step()                   # 4. atualiza os pesos
    if epoca % 5 == 0:
        l_te, a_te = avaliar(modelo, Xte_t, yte_t)
        print(f"época {epoca:2d}  perda teste = {l_te:.4f}  acurácia teste = {a_te:.3f}")

## Treinar bem

In [ ]:
def treinar(modelo, otimizador, X, y, X_val, y_val, epocas=100, batch=64,
            paciencia=None, seed=42):
    """Treina e devolve o histórico; com `paciencia`, restaura o melhor modelo (early stopping)."""
    carga = DataLoader(TensorDataset(X, y), batch_size=batch, shuffle=True,
                       generator=torch.Generator().manual_seed(seed))
    hist = {"perda_tr": [], "perda_val": [], "acc_tr": [], "acc_val": []}
    melhor, melhor_estado, sem_melhora = np.inf, None, 0
    for epoca in range(epocas):
        modelo.train()
        for xb, yb in carga:
            otimizador.zero_grad()
            nn.functional.cross_entropy(modelo(xb), yb).backward()
            otimizador.step()
        l_tr, a_tr = avaliar(modelo, X, y)
        l_val, a_val = avaliar(modelo, X_val, y_val)
        for k, v in zip(hist, (l_tr, l_val, a_tr, a_val)):
            hist[k].append(v)
        if paciencia is not None:
            if l_val < melhor - 1e-4:
                melhor, sem_melhora = l_val, 0
                melhor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
            else:
                sem_melhora += 1
                if sem_melhora >= paciencia:
                    break
    if melhor_estado is not None:
        modelo.load_state_dict(melhor_estado)
    hist["epocas_rodadas"] = len(hist["perda_tr"])
    return hist

In [ ]:
# poucos dados de treino (200) para provocar overfitting; validação separada do resto do treino
Xp, yp = Xtr_t[:200], ytr_t[:200]
Xv, yv = Xtr_t[200:600], ytr_t[200:600]

In [ ]:
torch.manual_seed(42)
grande = MLP(oculta=256)
h_sem = treinar(grande, torch.optim.Adam(grande.parameters(), lr=1e-3), Xp, yp, Xv, yv, epocas=150)
print(f"sem regularização: acc treino = {h_sem['acc_tr'][-1]:.3f} | acc validação = {h_sem['acc_val'][-1]:.3f}")
print(f"perda validação: mínima = {min(h_sem['perda_val']):.3f} (época {int(np.argmin(h_sem['perda_val']))+1}) | final = {h_sem['perda_val'][-1]:.3f}")

In [ ]:
torch.manual_seed(42)
reg = MLP(oculta=256, p_dropout=0.5)
h_reg = treinar(reg, torch.optim.AdamW(reg.parameters(), lr=1e-3, weight_decay=1e-2),
                Xp, yp, Xv, yv, epocas=150)
print(f"dropout + weight decay: acc treino = {h_reg['acc_tr'][-1]:.3f} | acc validação = {h_reg['acc_val'][-1]:.3f}")

torch.manual_seed(42)
es = MLP(oculta=256)
h_es = treinar(es, torch.optim.Adam(es.parameters(), lr=1e-3), Xp, yp, Xv, yv,
               epocas=150, paciencia=10)
print(f"early stopping: parou após {h_es['epocas_rodadas']} épocas | acc validação (melhor modelo) = {avaliar(es, Xv, yv)[1]:.3f}")

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(10, 3.6))
for h, rot in [(h_sem, "sem regularização"), (h_reg, "dropout + weight decay")]:
    eixos[0].plot(h["perda_val"], label=f"validação — {rot}")
    eixos[0].plot(h["perda_tr"], "--", label=f"treino — {rot}")
    eixos[1].plot(h["acc_val"], label=rot)
eixos[0].set_xlabel("época"); eixos[0].set_ylabel("perda"); eixos[0].legend(fontsize=7)
eixos[1].set_xlabel("época"); eixos[1].set_ylabel("acurácia de validação"); eixos[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
configs = {
    "SGD lr=0,01":            lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD lr=0,01 + momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam lr=1e-3":           lambda p: torch.optim.Adam(p, lr=1e-3),
    "Adam lr=1e-1 (alta)":    lambda p: torch.optim.Adam(p, lr=1e-1),
    "Adam lr=1e-5 (baixa)":   lambda p: torch.optim.Adam(p, lr=1e-5),
}
hists = {}
for nome, fabrica in configs.items():
    torch.manual_seed(42)
    m = MLP()
    hists[nome] = treinar(m, fabrica(m.parameters()), Xtr_t, ytr_t, Xte_t, yte_t, epocas=15)
    print(f"{nome:24s} perda treino final = {hists[nome]['perda_tr'][-1]:.4f} | acc teste = {hists[nome]['acc_val'][-1]:.3f}")

In [ ]:
plt.figure(figsize=(6, 3.6))
for nome, h in hists.items():
    plt.plot(h["perda_tr"], label=nome)
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("perda de treino"); plt.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Redes convolucionais

In [ ]:
Xtr_img = torch.tensor(X_tr / 16.0, dtype=torch.float32).reshape(-1, 1, 8, 8)
Xte_img = torch.tensor(X_te / 16.0, dtype=torch.float32).reshape(-1, 1, 8, 8)
print("formato (N, canais, altura, largura):", tuple(Xtr_img.shape))

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 8x8 -> 4x4
            nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2))  # 4x4 -> 2x2
        self.classificador = nn.Linear(16 * 2 * 2, 10)
    def forward(self, x):
        return self.classificador(self.conv(x).flatten(1))

torch.manual_seed(42)
cnn = CNN()
x = Xtr_img[:1]
for camada in cnn.conv:
    x = camada(x)
    print(f"{camada.__class__.__name__:10s} -> {tuple(x.shape)}")
print("parâmetros da CNN:", sum(p.numel() for p in cnn.parameters()))

In [ ]:
Xtr_flat, Xte_flat = Xtr_img.flatten(1), Xte_img.flatten(1)
res = {}
for nome, Modelo, (A, B) in [("MLP", MLP, (Xtr_flat, Xte_flat)), ("CNN", CNN, (Xtr_img, Xte_img))]:
    torch.manual_seed(42)
    modelo = Modelo()
    t0 = time.time()
    res[nome] = treinar(modelo, torch.optim.Adam(modelo.parameters(), lr=1e-3), A, ytr_t, B, yte_t, epocas=30)
    n_par = sum(p.numel() for p in modelo.parameters())
    print(f"{nome}: {n_par:5d} parâmetros | acc teste = {res[nome]['acc_val'][-1]:.3f} | {time.time()-t0:.1f} s")

In [ ]:
plt.figure(figsize=(6, 3.6))
for nome, h in res.items():
    plt.plot(h["acc_val"], label=nome)
plt.xlabel("época"); plt.ylabel("acurácia de teste"); plt.legend()
plt.tight_layout(); plt.show()

## Sequências: RNN e LSTM

In [ ]:
serie = serie_senoide(400, seed=42)
X_s, y_s = janelas(serie, tam=20)              # 20 passos passados -> próximo valor
n_tr = 300
Xs_tr, ys_tr = torch.tensor(X_s[:n_tr]), torch.tensor(y_s[:n_tr])
Xs_te, ys_te = torch.tensor(X_s[n_tr:]), torch.tensor(y_s[n_tr:])
print("janelas:", tuple(Xs_tr.shape), "->", tuple(ys_tr.shape))

In [ ]:
class PrevisorLSTM(nn.Module):
    def __init__(self, oculta=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=oculta, batch_first=True)
        self.saida = nn.Linear(oculta, 1)
    def forward(self, x):
        h, _ = self.lstm(x)                    # h: (lote, tempo, oculta)
        return self.saida(h[:, -1]).squeeze(-1)  # usa o último passo

torch.manual_seed(42)
prev = PrevisorLSTM()
opt = torch.optim.Adam(prev.parameters(), lr=1e-2)
for epoca in range(1, 151):
    prev.train(); opt.zero_grad()
    perda = nn.functional.mse_loss(prev(Xs_tr), ys_tr)
    perda.backward(); opt.step()
prev.eval()
with torch.no_grad():
    pred = prev(Xs_te)
mse_lstm = nn.functional.mse_loss(pred, ys_te).item()
mse_ingenuo = nn.functional.mse_loss(Xs_te[:, -1, 0], ys_te).item()   # repete o último valor
print(f"MSE teste LSTM = {mse_lstm:.4f} | MSE 'repetir o último valor' = {mse_ingenuo:.4f}")

In [ ]:
plt.figure(figsize=(8, 3.2))
plt.plot(ys_te.numpy(), label="real"); plt.plot(pred.numpy(), label="LSTM")
plt.xlabel("passo (conjunto de teste)"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
Xm, ym = tarefa_memoria(n=1200, T=20, seed=42)
Xm_tr, ym_tr = torch.tensor(Xm[:900]), torch.tensor(ym[:900])
Xm_te, ym_te = torch.tensor(Xm[900:]), torch.tensor(ym[900:])
print("a classe depende só do passo 0; os outros 19 passos são ruído")

In [ ]:
class Classificador(nn.Module):
    def __init__(self, tipo, oculta=16):
        super().__init__()
        self.tipo = tipo
        if tipo == "sem memória":
            self.rede = nn.Linear(1, 1)         # vê só o último passo
        else:
            camada = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[tipo]
            self.rec = camada(1, oculta, batch_first=True)
            self.rede = nn.Linear(oculta, 1)
    def forward(self, x):
        if self.tipo == "sem memória":
            return self.rede(x[:, -1]).squeeze(-1)
        h, _ = self.rec(x)
        return self.rede(h[:, -1]).squeeze(-1)

for tipo in ["sem memória", "RNN", "GRU", "LSTM"]:
    torch.manual_seed(42)
    m = Classificador(tipo)
    opt = torch.optim.Adam(m.parameters(), lr=1e-2)
    for epoca in range(100):
        m.train(); opt.zero_grad()
        nn.functional.binary_cross_entropy_with_logits(m(Xm_tr), ym_tr).backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)   # contém gradientes explosivos
        opt.step()
    m.eval()
    with torch.no_grad():
        acc = ((m(Xm_te) > 0).float() == ym_te).float().mean().item()
    print(f"{tipo:12s} acurácia de teste = {acc:.3f}")

## Autoencoders e GANs

In [ ]:
torch.manual_seed(42)
Xtr_ae, Xte_ae = Xtr_img.flatten(1), Xte_img.flatten(1)       # pixels em [0, 1]

class Autoencoder(nn.Module):
    def __init__(self, latente=8):
        super().__init__()
        self.codificador = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, latente))
        self.decodificador = nn.Sequential(nn.Linear(latente, 32), nn.ReLU(),
                                           nn.Linear(32, 64), nn.Sigmoid())
    def forward(self, x):
        return self.decodificador(self.codificador(x))

ae = Autoencoder()
opt = torch.optim.Adam(ae.parameters(), lr=1e-2)
carga = DataLoader(TensorDataset(Xtr_ae), batch_size=64, shuffle=True,
                   generator=torch.Generator().manual_seed(42))
for epoca in range(60):
    for (xb,) in carga:
        opt.zero_grad()
        nn.functional.mse_loss(ae(xb), xb).backward()
        opt.step()
ae.eval()
with torch.no_grad():
    rec = ae(Xte_ae)
mse_ae = nn.functional.mse_loss(rec, Xte_ae).item()

pca = PCA(n_components=8).fit(X_tr / 16.0)                    # mesma dimensão do gargalo
mse_pca = np.mean((pca.inverse_transform(pca.transform(X_te / 16.0)) - X_te / 16.0) ** 2)
print(f"MSE de reconstrução (teste): autoencoder = {mse_ae:.4f} | PCA(8) = {mse_pca:.4f}")

In [ ]:
fig, eixos = plt.subplots(2, 8, figsize=(9, 2.8))
for i in range(8):
    eixos[0, i].imshow(Xte_ae[i].reshape(8, 8), cmap="gray_r"); eixos[0, i].axis("off")
    eixos[1, i].imshow(rec[i].reshape(8, 8), cmap="gray_r");    eixos[1, i].axis("off")
eixos[0, 0].set_title("original", fontsize=8, loc="left"); eixos[1, 0].set_title("reconstruído", fontsize=8, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
torch.manual_seed(42)
reais = torch.tensor(amostras_gaussiana(2000, mu=4.0, sigma=1.25, seed=42))
G = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 1))     # ruído (4) -> amostra (1)
D = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1))     # amostra -> logit "real"
optG = torch.optim.Adam(G.parameters(), lr=2e-3)
optD = torch.optim.Adam(D.parameters(), lr=2e-3)
bce = nn.functional.binary_cross_entropy_with_logits
um, zero = torch.ones(128, 1), torch.zeros(128, 1)

for passo in range(3000):
    x_real = reais[torch.randint(0, len(reais), (128,))]
    x_fake = G(torch.randn(128, 4))
    # 1) discriminador: reais -> 1, falsas -> 0
    optD.zero_grad()
    (bce(D(x_real), um) + bce(D(x_fake.detach()), zero)).backward()
    optD.step()
    # 2) gerador: tenta fazer o discriminador dizer "real"
    optG.zero_grad()
    bce(D(x_fake), um).backward()
    optG.step()

with torch.no_grad():
    gerado = G(torch.randn(5000, 4))
print(f"reais:   média = {reais.mean():.2f}, desvio = {reais.std():.2f}")
print(f"gerados: média = {gerado.mean():.2f}, desvio = {gerado.std():.2f}")

## Boas práticas

In [ ]:
def fixar_sementes(seed=42):
    np.random.seed(seed); torch.manual_seed(seed)

def treino_rapido():
    fixar_sementes(42)
    m = MLP()
    treinar(m, torch.optim.Adam(m.parameters(), lr=1e-3), Xtr_t, ytr_t, Xte_t, yte_t, epocas=5)
    return avaliar(m, Xte_t, yte_t)

print("rodada 1:", treino_rapido())
print("rodada 2:", treino_rapido())

In [ ]:
m = MLP(p_dropout=0.5)
x1 = Xte_t[:1]
m.train()
print("modo train (dropout ativo) :", m(x1)[0, :3].detach().numpy().round(3), m(x1)[0, :3].detach().numpy().round(3))
m.eval()
with torch.no_grad():
    print("modo eval + no_grad        :", m(x1)[0, :3].numpy().round(3), m(x1)[0, :3].numpy().round(3))

In [ ]:
fixar_sementes(42)
m = MLP()
treinar(m, torch.optim.Adam(m.parameters(), lr=1e-3), Xtr_t, ytr_t, Xte_t, yte_t, epocas=5)
caminho = os.path.join(tempfile.gettempdir(), "mlp_digitos.pt")
torch.save(m.state_dict(), caminho)               # salva só os pesos

novo = MLP()                                      # recria a arquitetura...
novo.load_state_dict(torch.load(caminho, weights_only=True))   # ...e carrega os pesos
novo.eval()
print("acurácia original :", round(avaliar(m, Xte_t, yte_t)[1], 4))
print("acurácia recarregada:", round(avaliar(novo, Xte_t, yte_t)[1], 4))